In [1]:
import warnings
from copy import deepcopy

import pandas as pd
import gtfs_kit as gk
from typing import List, Tuple, Set
import copy
from pathlib import Path
import re

In [2]:
gtfs_output_path = Path("/home/simpal/otp/data")

gtfs_year = 2025
gtfs_root = Path("/home/simpal/O/sharing-trans-data/GTFS Data/CLEAN - GTFS DATA/" + str(gtfs_year))

if not gtfs_root.is_dir():
    print("INPUT ERROR: Directory not found: " + str(gtfs_root))
# Recursively find zip files
gtfs_files = list(gtfs_root.rglob("*.zip"))

# Extract YYYYMMDD from filename
def extract_date(path):
    match = re.search(r"\d{8}", path.name)
    return pd.to_datetime(match.group(), format="%Y%m%d") if match else None


gtfs_release = (
    pd.DataFrame({
        "path": gtfs_files,
        "file": [p.name for p in gtfs_files],
        "date": [extract_date(p) for p in gtfs_files],
    })
    .dropna(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
    .assign(
        date_end = lambda df: df["date"].shift(-1),
        date_days = lambda df: (df["date"] - pd.Timestamp("1970-01-01")).dt.days,
        date_end_days = lambda df: (df["date_end"] - pd.Timestamp("1970-01-01")).dt.days,
    )
)


print(f"Total number GTFS files: {len(gtfs_release)}")

Total number GTFS files: 24


In [3]:
gtfs_list = {}
#Read all gtfs_files
for _, row in gtfs_release.iterrows():
    print(f"Reading GTFS file: {row['file']}")

    feed = gk.feed.read_feed(row["path"], dist_units = "m") #Not sure what dist_unit is.

    gtfs_list[row["file"]] = feed


Reading GTFS file: GTFS_20250102.zip
Reading GTFS file: GTFS_20250113.zip
Reading GTFS file: GTFS_20250127.zip
Reading GTFS file: GTFS_20250210.zip
Reading GTFS file: GTFS_20250224.zip
Reading GTFS file: GTFS_20250310.zip
Reading GTFS file: GTFS_20250324.zip
Reading GTFS file: GTFS_20250407.zip
Reading GTFS file: GTFS_20250422.zip
Reading GTFS file: GTFS_20250505.zip
Reading GTFS file: GTFS_20250519.zip
Reading GTFS file: GTFS_20250616.zip
Reading GTFS file: GTFS_20250630.zip
Reading GTFS file: GTFS_20250728.zip
Reading GTFS file: GTFS_20250811.zip
Reading GTFS file: GTFS_20250825.zip
Reading GTFS file: GTFS_20250908.zip
Reading GTFS file: GTFS_20250922.zip
Reading GTFS file: GTFS_20251006.zip
Reading GTFS file: GTFS_20251020.zip
Reading GTFS file: GTFS_20251103.zip
Reading GTFS file: GTFS_20251117.zip
Reading GTFS file: GTFS_20251201.zip
Reading GTFS file: GTFS_20251215.zip


In [4]:
#truncating all files
from truncate_calendar_date import truncate_feed_to_date
for i, row in gtfs_release.iterrows():
    file_key = row["file"]
    cutoff = row["date_end"]

    if pd.isna(cutoff):
        continue

    print(f"Truncating {file_key} to {cutoff.date()}")
    gtfs_list[file_key] = truncate_feed_to_date(gtfs_list[file_key], cutoff)


Truncating GTFS_20250102.zip to 2025-01-13
Truncating GTFS_20250113.zip to 2025-01-27
Truncating GTFS_20250127.zip to 2025-02-10
Truncating GTFS_20250210.zip to 2025-02-24
Truncating GTFS_20250224.zip to 2025-03-10
Truncating GTFS_20250310.zip to 2025-03-24
Truncating GTFS_20250324.zip to 2025-04-07
Truncating GTFS_20250407.zip to 2025-04-22
Truncating GTFS_20250422.zip to 2025-05-05
Truncating GTFS_20250505.zip to 2025-05-19
Truncating GTFS_20250519.zip to 2025-06-16
Truncating GTFS_20250616.zip to 2025-06-30
Truncating GTFS_20250630.zip to 2025-07-28
Truncating GTFS_20250728.zip to 2025-08-11
Truncating GTFS_20250811.zip to 2025-08-25
Truncating GTFS_20250825.zip to 2025-09-08
Truncating GTFS_20250908.zip to 2025-09-22
Truncating GTFS_20250922.zip to 2025-10-06
Truncating GTFS_20251006.zip to 2025-10-20
Truncating GTFS_20251020.zip to 2025-11-03
Truncating GTFS_20251103.zip to 2025-11-17
Truncating GTFS_20251117.zip to 2025-12-01
Truncating GTFS_20251201.zip to 2025-12-15


In [5]:
#Merge all seperate feeds into one
print("\nMerging all feeds into combined GTFS feed...")
if "combined_feed" in globals(): #in case of rerun
    del combined_feed

feed_names = list(gtfs_list.keys())
combined_feed = copy.deepcopy(gtfs_list[feed_names[0]])
print(f"Starting with base feed: {feed_names[0]}")

# Add feed_id to the inital feed tables (same way you do for subsequent feeds)
for table in [
    "agency", "routes", "stops", "trips", "stop_times",
    "calendar", "calendar_dates", "shapes", "transfers"
]:
    df = getattr(combined_feed, table, None)
    setattr(combined_feed, table, df.assign(feed_id=feed_names[0]))

for feed_name in feed_names[1:]:
    print(f"Merging feed: {feed_name}")
    feed_to_merge = gtfs_list[feed_name]

    combined_feed.agency = pd.concat([combined_feed.agency, feed_to_merge.agency.assign(feed_id = feed_name)], ignore_index=True)
    combined_feed.routes = pd.concat([combined_feed.routes, feed_to_merge.routes.assign(feed_id = feed_name)], ignore_index=True)
    combined_feed.stops = pd.concat([combined_feed.stops, feed_to_merge.stops.assign(feed_id = feed_name)], ignore_index=True)
    combined_feed.trips = pd.concat([combined_feed.trips, feed_to_merge.trips.assign(feed_id = feed_name)], ignore_index=True)
    combined_feed.stop_times = pd.concat([combined_feed.stop_times, feed_to_merge.stop_times.assign(feed_id = feed_name)], ignore_index=True)
    combined_feed.calendar = pd.concat([combined_feed.calendar, feed_to_merge.calendar.assign(feed_id = feed_name)], ignore_index=True)
    combined_feed.calendar_dates = pd.concat([combined_feed.calendar_dates, feed_to_merge.calendar_dates.assign(feed_id = feed_name)], ignore_index=True)
    combined_feed.shapes = pd.concat([combined_feed.shapes, feed_to_merge.shapes.assign(feed_id = feed_name)], ignore_index=True)
    combined_feed.transfers = pd.concat([combined_feed.transfers, feed_to_merge.transfers.assign(feed_id = feed_name)], ignore_index=True)

    del gtfs_list[feed_name], feed_to_merge


print(f"\nMerge complete! Combined feed statistics:")
if combined_feed.agency is not None:
    print(f"  Agencies: {len(combined_feed.agency)}")
if combined_feed.routes is not None:
    print(f"  Routes: {len(combined_feed.routes)}")
if combined_feed.stops is not None:
    print(f"  Stops: {len(combined_feed.stops)}")
if combined_feed.trips is not None:
    print(f"  Trips: {len(combined_feed.trips)}")
if combined_feed.stop_times is not None:
    print(f"  Stop times: {len(combined_feed.stop_times)}")
if combined_feed.calendar is not None:
    print(f"  Calendar entries: {len(combined_feed.calendar)}")
if combined_feed.shapes is not None:
    print(f"  Shape points: {len(combined_feed.shapes)}")


Merging all feeds into combined GTFS feed...
Starting with base feed: GTFS_20250102.zip
Merging feed: GTFS_20250113.zip
Merging feed: GTFS_20250127.zip
Merging feed: GTFS_20250210.zip
Merging feed: GTFS_20250224.zip
Merging feed: GTFS_20250310.zip
Merging feed: GTFS_20250324.zip
Merging feed: GTFS_20250407.zip
Merging feed: GTFS_20250422.zip
Merging feed: GTFS_20250505.zip
Merging feed: GTFS_20250519.zip
Merging feed: GTFS_20250616.zip
Merging feed: GTFS_20250630.zip
Merging feed: GTFS_20250728.zip
Merging feed: GTFS_20250811.zip
Merging feed: GTFS_20250825.zip
Merging feed: GTFS_20250908.zip
Merging feed: GTFS_20250922.zip
Merging feed: GTFS_20251006.zip
Merging feed: GTFS_20251020.zip
Merging feed: GTFS_20251103.zip
Merging feed: GTFS_20251117.zip
Merging feed: GTFS_20251201.zip
Merging feed: GTFS_20251215.zip

Merge complete! Combined feed statistics:
  Agencies: 479
  Routes: 38643
  Stops: 892866
  Trips: 4392418
  Stop times: 103083671
  Calendar entries: 37747
  Shape points: 8

In [6]:
combined_feed_org = copy.deepcopy(combined_feed)

In [ ]:
# gtfs_list[feed_names[0]].trips.groupby('route_id') \
#     .apply(lambda df: df.duplicated(subset=['service_id'], keep=False).mean())
# combined_feed.trips.groupby('route_id') \
#     .apply(lambda df: df.duplicated(subset=['service_id'], keep = False).mean())
#
# combined_feed.trips['block_id'].isna().sum()/len(combined_feed.trips)

In [20]:
# #add prefix to non-unique service_id.
# # Rest ID are prefixed later (!), but since service_id is especially inconsistent across feed over time, prefix is added before trying to drop duplicates later.
#
# from prefix_conflicting_ids import apply_prefix_to_feed, find_conflicting_ids
# from id_configuration import ID_CONFIG, ID_CONFIG_service_id
#
# primary_table, config = next(iter(ID_CONFIG_service_id.items()))
# print(f"\nProcessing {primary_table}...")
# conflicting = find_conflicting_ids(
#     combined_feed,
#     config["id_col"],
#     primary_table,
#     config["identity_cols"]
# )
# print(f"  Found {len(conflicting)} conflicting {config["id_col"]} values")
# if conflicting:
#     apply_prefix_to_feed(combined_feed, config["id_col"], conflicting, config["foreign_keys"], primary_table)
#
#     print(f"    Prefixed conflicting {config["id_col"]} in all feeds")


Processing calendar...
  Found 1953 conflicting service_id values
    Prefixed conflicting service_id in all feeds


In [50]:
import prefix_ids
import importlib
import id_configuration
import deduplication

importlib.reload(prefix_ids)
importlib.reload(id_configuration)
importlib.reload(deduplication)

<module 'deduplication' from '/home/simpal/otp/gtfs_merger/deduplication.py'>

In [48]:
from prefix_ids import apply_prefix_to_all_ids
from id_configuration import ID_CONFIG, ID_CONFIG_service_id

for primary_table, config in ID_CONFIG.items():
    if primary_table in ['stops', 'calendar_dates', 'stop_times']:
        continue
    apply_prefix_to_all_ids(combined_feed, config["id_col"], config["foreign_keys"],
                         primary_table)
    print(f"    Prefixed conflicting {config["id_col"]} in all feeds")


    Prefixed conflicting shape_id in all feeds
    Prefixed conflicting agency_id in all feeds
    Prefixed conflicting route_id in all feeds
    Prefixed conflicting service_id in all feeds
    Prefixed conflicting trip_id in all feeds


In [52]:
from deduplication import deduplicate_feed

tables_to_deduplicate = ['stops', 'shapes', 'agency', 'routes', 'calendar', 'trips']

print("Starting deduplication process...")
print(f"\nBefore deduplication:")
for table in tables_to_deduplicate:
    if getattr(combined_feed, table, None) is not None:
        print(f"  {table}: {len(getattr(combined_feed, table, None))}")

# Apply deduplication for each ID type
for primary_table, config in ID_CONFIG.items():
    print(f"\nDeduplicating {primary_table}...")

    removed = deduplicate_feed(
        combined_feed,
        config["id_col"],
        primary_table,
        config["identity_cols"],
        config["foreign_keys"]
    )

    df = getattr(combined_feed, primary_table, None)
    if df is not None:
        print(f"  {primary_table}: removed {removed} duplicates, {len(df)} remaining")

print("\n" + "=" * 50)
print("Deduplication complete!")
print(f"\nAfter deduplication:")
for table in tables_to_deduplicate + ['stop_times']:
    df = getattr(combined_feed, table, None)
    if df is not None:
        print(f"  {table}: {len(df)}")

del df

Starting deduplication process...

Before deduplication:
  stops: 892866
  shapes: 89428529
  agency: 479
  routes: 38643
  calendar: 37747
  trips: 4392418

Deduplicating stops...
              stop_id
0       000461011300
1       000008600718
2       000008600719
3       000008600716
4       000008600717
...              ...
892861  000751439002
892862  000000056047
892863  000000056056
892864  000821001219
892865  000621470131

[38303 rows x 1 columns] 
Duplicated stop_id with lat/lon differences > 40, will get new stop_id.
  stops: removed 853897 duplicates, 38969 remaining

Deduplicating shapes...
  shapes: removed 69171194 duplicates, 20257335 remaining

Deduplicating agency...
  agency: removed 455 duplicates, 24 remaining

Deduplicating routes...
  routes: removed 36969 duplicates, 1674 remaining

Deduplication complete!

After deduplication:
  stops: 38969
  shapes: 20257335
  agency: 24
  routes: 1674
  calendar: 37747
  trips: 4392418
  stop_times: 103083671


In [ ]:
# # Apply to prefix to all ID types
# for primary_table, config in ID_CONFIG.items():
#     if config['primary_table'].isin(['calendar', 'calendar_dates', 'shapes', 'stop_times']):
#         continue
#     print(f"\nProcessing {primary_table}...")
#     conflicting = find_conflicting_ids(
#         combined_feed,
#         config["id_col"],
#         primary_table,
#         config["identity_cols"]
#     )
#     print(f"  Found {len(conflicting)} conflicting {config["id_col"]} values")
#     if conflicting:
#         apply_prefix_to_feed(combined_feed, config["id_col"], conflicting, config["foreign_keys"], config["primary_table"])
#         print(f"    Prefixed conflicting {config["id_col"]} in all feeds")